# 07 — Evaluation

**This notebook answers evaluator points 1, 2, 3, 4, 6, 7 and 8 in one place.**

| Point | Where it is answered |
|---|---|
| 1. Precision | section 2 |
| 2. Recall | section 2 |
| 3. F1-score | section 2 |
| 4. False-negative rate | section 2, reported explicitly as "we miss X%" |
| 6. Forecast horizons 1 h / 3 h / 6 h | every table is split by horizon |
| 7. Time-based train/test split | section 1 — rolling origin, four folds |
| 8. False-positive / false-negative analysis | sections 5 and 6 |

**Three principles this notebook sticks to.**

1. **No accuracy.** At a base rate near 1 in 4,000, predicting "no flood" always gives
   99.97% accuracy. The number is worse than useless — it is actively misleading.
2. **Every headline gets decomposed.** Overall recall is always shown next to onset recall,
   because the gap between them is the difference between forecasting and monitoring.
3. **Thresholds come from validation, never from test.** Tuning a cut-off on the data you
   report is how honest people accidentally overstate results by ten points.

In [ ]:
import sys, pathlib
# Make the shared library importable no matter where Jupyter was started from.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from bkkflood import CFG, PATHS
print("project root:", ROOT)
print("config version:", CFG["project"]["version"])
import json
import matplotlib.pyplot as plt
from bkkflood.metrics import (classification_metrics, decompose_by_onset, threshold_sweep,
                              pick_threshold, per_station_breakdown, extract_cases,
                              event_scores, coverage)
from bkkflood.modeling import (load_model, persistence_score, hybrid_score,
                               fit_calibrator, calibration_table)
from bkkflood.splits import rolling_origin_folds, split_frame, summarise_folds, holdout_is_sealed
from bkkflood.features import feature_columns, load_training_years
from bkkflood.labels import find_events_frame

PRIMARY = CFG["flood_event"]["primary_tier_cm"]
HORIZONS = CFG["forecast"]["horizons_h"]

## 1. The evaluation protocol, stated before any numbers appear

Say what you did before you say how well it went. A reader who disagrees with the protocol
can stop reading here rather than being persuaded by numbers produced under rules they
would have rejected.

In [ ]:
protocol = {
    "split": "rolling origin, chronological, no shuffling",
    "folds": [f.describe() for f in rolling_origin_folds()],
    "embargo_hours": CFG["splits"]["rolling_origin"]["embargo_hours"],
    "threshold_selection": "validation year only, then frozen for test",
    "primary_tier_cm": PRIMARY,
    "horizons_h": HORIZONS,
    "flood_event": CFG["flood_event"],
    "final_holdout_sealed": holdout_is_sealed(),
}
print(json.dumps(protocol, indent=2, default=str))

if not holdout_is_sealed():
    print("\nNOTE: the final holdout has already been opened by an earlier version of "
          "this project.\nRolling-origin folds are therefore the headline evidence, and "
          "any number quoted\nfrom the holdout year must be labelled as reused.")

In [ ]:
# Load per fold, not all at once — see the memory note in notebook 04.
from bkkflood.data import load_years, feature_list, available_years

YEARS = available_years()
FEATURES = feature_list()
print(f"years built: {YEARS}")
print(f"{len(FEATURES)} features")

## 2. The headline table

Precision, recall, F1, F2 and the false-negative rate, for every tier and horizon, averaged
across folds with the spread shown.

In [ ]:
results_path = PATHS.reports / "lightgbm_fold_results.csv"
if results_path.exists():
    folds_df = pd.read_csv(results_path)
else:
    folds_df = pd.DataFrame()
    print("Run notebook 04 first.")

if len(folds_df):
    summary = summarise_folds(
        folds_df.to_dict("records"),
        metric_cols=["precision", "recall", "f1", "f2", "false_negative_rate",
                     "pr_auc", "onset_recall"],
        group_cols=["target", "horizon_h"])
    show = summary[["target", "horizon_h", "recall_mean", "recall_std",
                    "precision_mean", "f1_mean", "f2_mean",
                    "false_negative_rate_mean", "onset_recall_mean"]].round(3)
    display(show)

In [ ]:
# The same numbers written the way a person would say them out loud.
if len(folds_df):
    primary = folds_df[folds_df["target"] == f"ge{PRIMARY}"]
    print(f"At the {PRIMARY} cm tier, averaged over "
          f"{primary['fold'].nunique()} independent test years:\n")
    for horizon in HORIZONS:
        sub = primary[primary["horizon_h"] == horizon]
        if sub.empty:
            continue
        r, p = sub["recall"].mean(), sub["precision"].mean()
        onset = sub["onset_recall"].mean() if "onset_recall" in sub else float("nan")
        print(f"  {horizon}h ahead:")
        print(f"     catches {r:.0%} of flood rows, misses {1 - r:.0%}")
        print(f"     {p:.0%} of alarms are real -> about "
              f"{(1 - p) / max(p, 1e-9):.1f} false alarms per true one")
        print(f"     on rows that were NOT already flooded, catches {onset:.0%}")
        print(f"     recall range across years: {sub['recall'].min():.2f} "
              f"to {sub['recall'].max():.2f}\n")

**On the recall range.** A metric that swings from 0.18 to 0.44 across years should
never be summarised as "0.31" without that range beside it. The swing is not noise in the
model — it is the difference between a wet year and a dry one, and an operator planning
staffing needs to know it exists.

## 3. The false-negative rate, stated as a miss rate

FNR is `1 - recall`, but the two phrasings land completely differently on a reader. "Recall
0.55" sounds like a result. "We miss 45% of floods" is the same fact and is the one that
prompts the right follow-up question, which is: *which* 45%?

In [ ]:
if len(folds_df):
    fnr = (folds_df.groupby(["target", "horizon_h"])["false_negative_rate"]
                   .agg(["mean", "min", "max"]).round(3))
    fnr.columns = ["miss_rate", "best_year", "worst_year"]
    display(fnr)

    ceiling = CFG["operating_point"]["max_false_negative_rate"]
    breaches = fnr[fnr["miss_rate"] > ceiling]
    print(f"Configured ceiling on the miss rate: {ceiling:.0%}")
    if len(breaches):
        print(f"Above the ceiling in {len(breaches)} tier-horizon combinations:")
        display(breaches)
        print("This is an honest limitation, not a bug. At 6 hours the model has very "
              "little to work with: our rainfall input is a district average and we have "
              "no radar. Report it plainly rather than lowering the threshold until the "
              "number looks acceptable — that would just trade misses for false alarms.")

## 4. Is the risk percentage honest?

The dashboard shows a number like "68% risk". If rows predicted at 68% flood 4% of the
time, that display is lying to an operator, and the fix is calibration, not a disclaimer.

Downsampling during training guarantees the raw scores are distorted, so this step is not
optional.

In [ ]:
# Calibration is fitted on VALIDATION and checked on test.
#
# Fitting it on training data would reproduce the training distribution rather
# than the real one, which defeats the purpose. Neither split is downsampled —
# calibration against a distorted class balance would be worse than none.
#
# `val` and `test` are reused by every section below, so the depth labels are
# read here alongside the tier labels: section 8 scores the P05-P95 bands
# against y_maxdepth_*, and would otherwise raise KeyError on a column that was
# never loaded.
fold = rolling_origin_folds()[-1]
label_cols = ([f"y_ge{PRIMARY}_{h}h" for h in HORIZONS]
              + [f"y_maxdepth_{h}h" for h in HORIZONS])
onset_col = f"is_onset_ge{PRIMARY}"

val = load_years([fold.val_year], features=FEATURES, labels=label_cols,
                 extra=[onset_col], verbose=False)
test = load_years([fold.test_year], features=FEATURES, labels=label_cols,
                  extra=[onset_col], verbose=False)
print(f"val {len(val):,} rows, test {len(test):,} rows")

calibrators = {}
for horizon in HORIZONS:
    label = f"y_ge{PRIMARY}_{horizon}h"
    stem = f"clf_ge{PRIMARY}_{horizon}h"
    try:
        model = load_model(stem)
    except Exception as exc:
        print(f"{stem}: not saved yet ({exc})")
        continue

    val_scores = model.predict(val)
    calibrator = fit_calibrator(val_scores, val[label].to_numpy())
    calibrators[stem] = calibrator

    raw_table = calibration_table(model.predict(test), test[label].to_numpy())
    cal_table = calibration_table(calibrator.predict(model.predict(test)),
                                  test[label].to_numpy())
    print(f"\n--- {horizon}h ---")
    print("raw scores:")
    print(raw_table.to_string(index=False))
    print("after calibration:")
    print(cal_table.to_string(index=False))

In [ ]:
# Reliability diagram: predicted risk against what actually happened.
if calibrators:
    fig, axes = plt.subplots(1, len(calibrators), figsize=(5 * len(calibrators), 4),
                             squeeze=False)
    for ax, (stem, calibrator) in zip(axes[0], calibrators.items()):
        horizon = int(stem.split("_")[-1].rstrip("h"))
        model = load_model(stem)
        label = f"y_ge{PRIMARY}_{horizon}h"
        table = calibration_table(calibrator.predict(model.predict(test)),
                                  test[label].to_numpy(), n_bins=12)
        ax.plot([0, table["mean_predicted"].max()], [0, table["mean_predicted"].max()],
                "k--", label="perfect")
        ax.plot(table["mean_predicted"], table["observed_rate"], "o-", label="model")
        ax.set_xlabel("predicted risk"); ax.set_ylabel("observed rate")
        ax.set_title(f"{horizon}h reliability"); ax.legend()
    plt.tight_layout(); plt.show()

    import joblib
    joblib.dump(calibrators, PATHS.artifacts / "calibrators.joblib")
    print(f"Saved calibrators -> {PATHS.artifacts / 'calibrators.joblib'}")

**Reading a reliability diagram.** Points on the dashed line mean the stated risk is
the real risk. Points below it mean the model overstates danger; points above mean it
understates it. Understating is the worse failure here, and it is the one to check first.

## 5. Where the misses are — per-station analysis

Errors are almost never spread evenly. Usually a handful of stations produce most of them,
and they usually have a physical explanation that no amount of tuning will fix. Finding
them turns "the model is 55% accurate" into a specific, actionable list.

In [ ]:
horizon = 1
label = f"y_ge{PRIMARY}_{horizon}h"
stem = f"clf_ge{PRIMARY}_{horizon}h"

try:
    model = load_model(stem)
    scored = test.copy()
    scored["score"] = hybrid_score(model.predict(test), persistence_score(test, PRIMARY))
    scored["pred"] = (scored["score"] >= (model.threshold or 0.5)).astype(int)

    by_station = per_station_breakdown(scored, label, "pred")
    print(f"Stations producing the most missed floods ({horizon}h, {PRIMARY} cm):")
    display(by_station.head(15))

    total_fn = by_station["fn"].sum()
    top10 = by_station.head(10)["fn"].sum()
    print(f"\nTop 10 stations account for {top10}/{total_fn} misses "
          f"({top10 / max(total_fn, 1):.0%}) across {len(by_station)} stations.")
except Exception as exc:
    print(f"Train and save the models first (notebook 04). {exc}")

In [ ]:
# The same view for false alarms — the ones that erode operator trust.
if "by_station" in dir():
    noisy = by_station.sort_values("fp", ascending=False).head(10)
    print("Stations producing the most false alarms:")
    display(noisy[["station_code", "fp", "tp", "precision", "positives"]])
    print("A station with many false alarms and almost no real events is usually a "
          "faulty sensor rather than a modelling problem. Check it against the quality "
          "scorecard from notebook 01 before touching the model.")

## 6. Individual cases — read the actual mistakes

Aggregate metrics tell you how much you are wrong. Reading individual failures tells you
*why*, and that is what changes what you build next.

In [ ]:
if "scored" in dir():
    context = [c for c in ["fl_depth_now", "rain_rf1hr_mean", "rain_rf3hr_mean",
                           "water_rising_share", "rain_fcst_3h"] if c in scored.columns]

    print("=== The most confident misses (model was sure it was safe) ===")
    misses = extract_cases(scored, label, "pred", "score", kind="fn",
                           top_n=10, context_cols=context)
    display(misses)

    print("\n=== The loudest false alarms ===")
    alarms = extract_cases(scored, label, "pred", "score", kind="fp",
                           top_n=10, context_cols=context)
    display(alarms)

**How to read a miss.** Look at the rain columns. Two very different failures hide in
the same statistic:

* **Rain was falling and we still missed it.** A model problem — the signal was present and
  we did not use it. Worth fixing.
* **No rain anywhere and the water arrived anyway.** Not a model problem. Either the rain
  fell between gauges (the district-average weakness again), or the water came from
  somewhere else entirely — an upstream canal, a pump switching off, a blocked drain. No
  amount of tuning fixes this. It needs radar, canal topology, or pump records.

Sorting the misses into those two buckets is the single most useful hour you can spend on
this project, and it is what turns the evaluation into a data request.

In [ ]:
# Sort the misses into those two buckets automatically.
if "misses" in dir() and "rain_rf1hr_mean" in misses.columns:
    rainless = misses["rain_rf1hr_mean"].fillna(0) < 1.0
    print(f"Of the {len(misses)} worst misses:")
    print(f"  {rainless.sum()} happened with essentially no rain recorded nearby")
    print(f"  {(~rainless).sum()} happened while rain was falling")
    print()
    print("The rainless ones point at missing data (radar, canal topology, pumps).")
    print("The rainy ones point at the model and are the ones worth working on.")

## 7. Event-level scoring — what an operator experiences

Row-level metrics over-weight long floods: a six-hour event contributes 24 rows, a
20-minute one contributes 1. Operators do not think in rows. They think in events, and in
how much warning they got.

In [ ]:
events_path = PATHS.reports / "flood_events_15cm.csv"
if events_path.exists() and "scored" in dir():
    events = pd.read_csv(events_path, parse_dates=["start", "end", "peak_time"])
    test_year = fold.test_year
    test_events = events[events["start"].dt.year == test_year]

    # Named alarm_rows, not alarms: section 6 binds `alarms` to the worst false
    # positives and section 9 writes that variable out as
    # worst_false_positives.csv. Reusing the name here would silently replace
    # those ten diagnostic rows with every firing in the test year.
    alarm_rows = scored[scored["pred"] == 1][["station_code", "site_timestamp"]]
    scores = event_scores(test_events, alarm_rows, horizon_h=horizon,
                          tolerance_min=CFG["forecast"]["event_match_tolerance_min"])

    print(f"Event-level results for {test_year} ({horizon}h model):")
    for key, value in scores.items():
        print(f"  {key:22s} {value}")
    print()
    print(f"In plain terms: of {scores['events']} flood events, we warned about "
          f"{scores['caught']} ({scores['pod']:.0%}), with a median of "
          f"{scores['median_lead_minutes']} minutes of warning.")

**Why event POD usually beats row recall.** A long flood only has to be caught once to
count as caught. That is the right way to score it: an operator who gets one alarm at the
start of a three-hour flood has been warned, even if the model then goes quiet.

**Median lead time is the number to put in front of BMA.** "63% of floods, with a median
25 minutes of warning" is something a duty officer can plan around. "PR-AUC 0.56" is not.

## 8. Depth bands — is the uncertainty honest?

In [ ]:
for horizon in HORIZONS:
    try:
        lo = load_model(f"reg_q05_{horizon}h")
        hi = load_model(f"reg_q95_{horizon}h")
    except Exception:
        continue
    truth = test[f"y_maxdepth_{horizon}h"].to_numpy()
    cov = coverage(truth, lo.predict(test), hi.predict(test))
    print(f"  {horizon}h: P05-P95 band contains {cov:.1%} of outcomes (target ~90%)")
    if cov < 0.85:
        print("     -> band is too narrow; the dashboard is understating the worst case")
    elif cov > 0.95:
        print("     -> band is too wide; it will look uninformative to an operator")

## 9. Save everything the report needs

In [ ]:
artefacts = {}
if len(folds_df):
    artefacts["fold_summary"] = summarise_folds(
        folds_df.to_dict("records"),
        group_cols=["target", "horizon_h"])
    artefacts["fold_summary"].to_csv(PATHS.reports / "fold_summary.csv", index=False)
if "by_station" in dir():
    by_station.to_csv(PATHS.reports / "station_breakdown.csv", index=False)
if "misses" in dir():
    misses.to_csv(PATHS.reports / "worst_false_negatives.csv", index=False)
if "alarms" in dir():
    alarms.to_csv(PATHS.reports / "worst_false_positives.csv", index=False)

(PATHS.reports / "evaluation_protocol.json").write_text(
    json.dumps(protocol, indent=2, default=str))
print(f"Wrote evaluation outputs -> {PATHS.reports}")
for p in sorted(PATHS.reports.glob("*")):
    print(f"  {p.name}")

### The paragraph to put in the report

> Evaluated with rolling-origin cross-validation across four chronological folds
> (2022-2025 as successive test years), with thresholds selected on the validation year and
> frozen before testing. At the 15 cm advisory tier the model achieves recall of R (range
> across years), precision P, and a false-negative rate of F — that is, it misses F of
> flood rows. Decomposed, recall on already-flooded rows is near-total (a persistence rule
> achieves the same), while recall on genuine onset rows is O; the onset specialists in
> notebook 05 raise that figure substantially at short range. At the event level the system
> warns about E% of flood episodes with a median lead time of L minutes.
>
> Skill decays sharply with horizon. At 6 hours the model is close to climatology, because
> every rainfall input is a district average of rain that has already fallen. Radar
> rainfall and archived forecast rainfall are the two inputs most likely to change that.

Next: `08_final_holdout.ipynb`.